# NISAR L2 GUNW (Level-2 Geocoded Unwrapped and Wrapped phase) Product Tutorial

This tutorial provides a guide to working with NISAR GUNW products, including accessing, reading, and analyzing the data.

**Author:** Heresh Fattahi, July 2025

## Table of Contents
1. [Introduction to GUNW](#1-introduction-to-gunw)
2. [Granule Naming Convention](#2-granule-naming-convention)
3. [Download a GUNW Product from ASF DAAC](#3-download-a-gunw-product-from-asf-daac)
4. [Explore the HDF5 Structure](#4-explore-the-hdf5-structure)
5. [Main Datasets in GUNW](#5-main-datasets-in-gunw)
6. [Correction Layers](#6-correction-layers)
7. [Unwrapping Quality Metric: Connected Components](#7-unwrapping-quality-metric-connected-components)
8. [Mask Layer](#8-mask-layer)
9. [Pixel Offsets](#9-pixel-offsets)
10. [Metadata Cube](#10-metadata-cube)
11. [Phase Sign Convention](#11-phase-sign-convention)
12. [Processing Information](#12-processing-information)
13. [QA Report and Browse](#13-qa-report-and-browse)

## Setup and Import Required Libraries

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from osgeo import gdal, osr
import warnings
warnings.filterwarnings('ignore')

# Set display options
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

## 1. Introduction to GUNW

### What is GUNW?

The **GUNW (Geocoded Unwrapped Interferogram)** product is a Level-2 (L2) product. It provides geocoded, multi-looked, ellipsoid and topography-flattened unwrapped interferometric phase and coherence at 80 m posting with several correction layers and with an additional geocoded wrapped phase at 20 m posting.

### Key Features:

- **Product Level**: Level 2 (geocoded and map-projected)
- **Format**: HDF5 (Hierarchical Data Format version 5)
- **Coordinate System**: UTM or Polar Stereographic projection
- **Spatial Coverage**: Pre-defined track/frame (typically 240 km × 240 km)
- **Temporal Baseline**: Nearest pair in time
- **Polarization**: Co-pol channels only (HH or VV)

### What's Inside GUNW?

1. **Geocoded Unwrapped interferometric phase** (80 m posting)
2. **Geocoded Wrapped complex interferogram** (20 m posting)
3. **Coherence magnitude** (20 m and 80 m posting)
4. **Connected components mask** (unwrapping quality)
5. **Mask**: water mask, valid data mask of input reference and secondary RSLCs, data exception mask, ionosphere mask
7. **Pixel offsets** (along-track and slant range, 80 m posting)
8. **Correction layers** (ionosphere, troposphere, solid Earth tides)
9. **Metadata cubes** (incidence angle, baseline, etc.)

### Product Dependency

GUNW is derived from:
- **RIFG** (Range-Doppler Interferogram)
- **RUNW** (Range-Doppler Unwrapped Interferogram)

Which in turn are derived from:
- **RSLC** (Range-Doppler Single Look Complex) products

## 2. Granule Naming Convention

NISAR GUNW products follow the NISAR Standard Product File Naming Scheme (JPL D-102255).

### Naming Format:

```
NISAR_IL_PT_PROD_CYL_REL_P_FRM_SCY_MODE_PO_RefStartDateTime_RefEndDateTime_SecStartDateTime_SecEndDateTime_CRID_A_C_LOC_CTR.EXT
```

Where:

* **`NISAR`** – 5 characters for mission: `nisar`
* **`I`** – 1 character for instrument:
    * `L` for L-SAR
    * `S` for S-SAR
* **`L`** – 1 character for processing level: `1` or `2`
* **`PT`** – 2 characters for processing type:
    * `PR` – Production
    * `UR` – Urgent response
    * `OD` – Science on-demand
* **`PROD`** – 4 characters for product ID: `rifg`, `runw`, `gunw`, `roff`, `goff`
* **`CYL`** – 3 characters for cycle number in the mission. Each cycle represents 12 days, zero-padded, starting at `001` [rd1].
* **`REL`** – 3 characters for relative orbit track number within a cycle. Resets to `1` with a cycle number increment, zero-padded. Valid values: `001`-`173` [rd1].
* **`P`** – 1 character for direction of movement of the satellite at the time of imaging [rd1]:
    * `A` for ascending
    * `D` for descending
* **`FRM`** – 3 characters for track frame number, a segment of an orbital track corresponding to the product, zero-padded. Valid values: `001`-`176` on each track (see sec 4.5).
* **`SCY`** – 3 characters for the second cycle number.
* **`MODE`** – 4 characters for bandwidth mode code of primary and secondary:
    * `40`, `20`, `77`, `05`, or `00` (only if the secondary band is missing)
* **`PO`** – 2 characters for polarization:
    * *Note: The polarization pair notation refers to the transmit and receive polarizations, respectively, of the main band and the side band of the reference (e.g., hv refers to h transmit and v receive).*
    * `SV` = VV – single polarity
    * `SH` = HH – single polarity
    * `DH` = HH/HV – dual polarity
    * `DV` = VV/VH – dual polarity
    * `CL` = LH/LV – compact polarity
    * `CR` = RH/RV – compact polarity
    * `QP` = HH/HV/VV/VH – quad polarity
    * `QH` = HH/HV + VV/VH – quasi quad polarity
    * `QV` = VV/VH + HH/HV – quasi quad polarity
    * `QD` = HH/VV – quasi dual pole
* **`RefStartDateTime`** – 15 characters for the data from the reference listed cycle contained in the file as `yyyymmddThhmmss`, UTC.
* **`RefEndDateTime`** – 15 characters for the data from the reference listed cycle contained in the file as `yyyymmddThhmmss`, UTC.
* **`SecStartDateTime`** – 15 characters for the data from the secondary listed cycle contained in the file as `yyyymmddThhmmss`, UTC.
* **`SecEndDateTime`** – 15 characters for the data from the secondary listed cycle contained in the file as `yyyymmddThhmmss`, UTC.


## 3. Download a GUNW Product from ASF DAAC

NISAR data products are distributed through the **Alaska Satellite Facility (ASF) Distributed Active Archive Center (DAAC)**.

### Methods to Access Data:

1. **ASF Data Search (Vertex)**: https://search.asf.alaska.edu/
2. **ASF Data Search API**: Programmatic access
3. **Earthdata Search**: https://search.earthdata.nasa.gov/

### Using Python to Download (Example):

In [ ]:
# Example: Download GUNW product using asf_search
# Note: This requires authentication with NASA Earthdata Login

# Uncomment to install asf_search if needed:
# !pip install asf_search

# import asf_search as asf

# # Set up authentication
# session = asf.ASFSession().auth_with_creds('username', 'password')

# # Search for GUNW products
# results = asf.search(
#     dataset=asf.DATASET.NISAR,
#     processingLevel='L2_GUNW',
#     start='2025-03-01',
#     end='2025-03-31'
# )

# # Download the first result
# results[0].download(path='./', session=session)

# For this tutorial, we'll use a sample file path
gunw_file = 'NISAR_L2_PR_GUNW_024_013_D_071_025_4000_SH_20260627T024036_20260627T024114_20260709T024036_20260709T024113_P05023_N_F_J_001.h5'

print(f"GUNW file: {gunw_file}")

## 4. Explore the HDF5 Structure

NISAR products are stored in HDF5 format, which provides:
- Hierarchical organization (similar to a file system)
- Efficient storage and access
- Self-describing metadata
- Cross-platform compatibility

### HDF5 Key Concepts:

- **File**: Container for all data
- **Groups**: Like directories, organize data hierarchically
- **Datasets**: Arrays of data
- **Attributes**: Metadata attached to Groups or Datasets
- **Datatypes**: Define the format of data elements

In [ ]:
def explore_hdf5_structure(filename, max_depth=5):
    """
    Recursively explore and print HDF5 file structure
    """
    def print_structure(name, obj, depth=0):
        if depth > max_depth:
            return
        
        indent = '  ' * depth
        if isinstance(obj, h5py.Group):
            print(f"{indent}📁 {name}/")
        elif isinstance(obj, h5py.Dataset):
            print(f"{indent}📄 {name} {obj.shape} {obj.dtype}")
    
    with h5py.File(filename, 'r') as f:
        print(f"\nHDF5 File: {filename}\n")
        print("=" * 80)
        f.visititems(print_structure)

explore_hdf5_structure(gunw_file, max_depth=4)

### GUNW HDF5 Group Organization

```
/
└── science/
    └── LSAR/
        ├── identification/          # Product identification metadata
        └── GUNW/
            ├── grids/
            │   └── frequencyA/      # Main imaging band (77 MHz)
            │       ├── wrappedInterferogram/
            │       │   ├── HH/      # or VV
            │       │   │   ├── wrappedInterferogram
            │       │   │   ├── coherenceMagnitude
            │       │   │   ├── xCoordinates
            │       │   │   ├── yCoordinates
            │       │   │   └── projection
            │       ├── unwrappedInterferogram/
            │       │   ├── HH/      # or VV
            │       │   │   ├── unwrappedPhase
            │       │   │   ├── coherenceMagnitude
            │       │   │   ├── connectedComponents
            │       │   │   ├── ionospherePhaseScreen
            │       │   │   ├── ionospherePhaseScreenUncertainty
            │       │   │   ├── mask
            │       │   │   └── ...
            │       └── pixelOffsets/
            │           └── HH/      # or VV
            │               ├── alongTrackOffset
            │               ├── slantRangeOffset
            │               └── correlationSurfacePeak
            └── metadata/
                ├── processingInformation/
                │   ├── parameters/
                │   ├── algorithms/
                │   └── inputs/
                ├── orbit/
                ├── attitude/
                └── radarGrid/       # Metadata cubes
```

In [ ]:
# Function to read global attributes
def print_global_attributes(filename):
    """
    Print global attributes of GUNW product
    """
    with h5py.File(filename, 'r') as f:
        print("\nGlobal Attributes:")
        print("=" * 80)
        for key, value in f.attrs.items():
            print(f"{key:30s}: {value}")

# Example usage:
print_global_attributes(gunw_file)

In [ ]:
# Function to read identification metadata
def print_identification_info(filename):
    """
    Print product identification information
    """
    with h5py.File(filename, 'r') as f:
        id_group = f['/science/LSAR/identification']
        fields = f['/science/LSAR/identification'].keys()
        print("\nProduct Identification:")
        print("=" * 80)
        
        # Print important identification fields
        #fields = [
        #    'productVersion',
        #    'processingDateTime',
        #    'missionId',
        #    'trackNumber',
        #    'frameNumber',
        #    'orbitPassDirection',
        #    'lookDirection'
        #]
        
        for field in fields:
            if field in id_group:
                value = id_group[field][()]
                if isinstance(value, bytes):
                    value = value.decode('utf-8')
                print(f"{field:30s}: {value}")

# Example usage:
print_identification_info(gunw_file)

## 5. Main Datasets in GUNW

The GUNW product contains several key datasets organized by resolution:

### 5.1 Unwrapped Phase (80 m posting)

**Path:** `/science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/[HH/VV]/unwrappedPhase`

- **Units:** Radians
- **Data Type:** float32
- **Grid Spacing:** 80 m
- **Description:** Geocoded, multi-looked, ellipsoid and topography-flattened unwrapped interferometric phase
- **Sign Convention:** 
  - **Positive phase** → Range increase (moving away from sensor)
  - **Negative phase** → Range decrease (moving toward sensor)

In [ ]:
def read_unwrapped_phase(filename, polarization='HH'):
    """
    Read and display unwrapped phase
    """
    with h5py.File(filename, 'r') as f:
        # Read unwrapped phase
        phase_path = f'/science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/{polarization}/unwrappedPhase'
        phase = f[phase_path][:]
        
        # Read coordinates
        x_coords = f[f'/science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/{polarization}/xCoordinates'][:]
        y_coords = f[f'/science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/{polarization}/yCoordinates'][:]
        
        # Read attributes
        attrs = dict(f[phase_path].attrs)
        
    # Plot
    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    
    im = ax.imshow(phase, cmap='jet', aspect='auto',
                   extent=[x_coords[0], x_coords[-1], y_coords[-1], y_coords[0]])
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
    ax.set_title(f'Unwrapped Phase - {polarization} Polarization')
    
    cbar = plt.colorbar(im, ax=ax, orientation='vertical', pad=0.02)
    cbar.set_label('Phase (radians)', rotation=270, labelpad=20)
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    valid_phase = phase[~np.isnan(phase)]
    print(f"\nUnwrapped Phase Statistics ({polarization}):")
    print(f"  Shape: {phase.shape}")
    print(f"  Min: {np.nanmin(phase):.3f} rad")
    print(f"  Max: {np.nanmax(phase):.3f} rad")
    print(f"  Mean: {np.nanmean(phase):.3f} rad")
    print(f"  Std: {np.nanstd(phase):.3f} rad")
    print(f"  Valid pixels: {len(valid_phase)} / {phase.size} ({100*len(valid_phase)/phase.size:.1f}%)")
    
    return phase, x_coords, y_coords

# Example usage:
phase, x, y = read_unwrapped_phase(gunw_file, polarization='HH')

### 5.2 Coherence Magnitude (80 m posting)

**Path:** `/science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/[HH/VV]/coherenceMagnitude`

- **Units:** Dimensionless (0-1)
- **Data Type:** float32
- **Grid Spacing:** 80 m
- **Description:** Normalized interferometric coherence magnitude
- **Interpretation:**
  - **1.0**: Perfect coherence
  - **0.0**: Complete decorrelation

In [ ]:
def read_coherence(filename, polarization='HH', posting='80m'):
    """
    Read and display coherence magnitude
    """
    with h5py.File(filename, 'r') as f:
        # Choose path based on posting
        if posting == '80m':
            base_path = f'/science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/{polarization}'
        else:  # 20m
            base_path = f'/science/LSAR/GUNW/grids/frequencyA/wrappedInterferogram/{polarization}'
        
        coherence = f[f'{base_path}/coherenceMagnitude'][:]
        x_coords = f[f'{base_path}/xCoordinates'][:]
        y_coords = f[f'{base_path}/yCoordinates'][:]
    
    # Plot
    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    
    im = ax.imshow(coherence, cmap='gray', vmin=0, vmax=1, aspect='auto',
                   extent=[x_coords[0], x_coords[-1], y_coords[-1], y_coords[0]])
    ax.set_xlabel('Easting (m)')
    ax.set_ylabel('Northing (m)')
    ax.set_title(f'Coherence Magnitude - {polarization} ({posting} posting)')
    
    cbar = plt.colorbar(im, ax=ax, orientation='vertical', pad=0.02)
    cbar.set_label('Coherence', rotation=270, labelpad=20)
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    valid_coh = coherence[~np.isnan(coherence)]
    print(f"\nCoherence Statistics ({polarization}, {posting}):")
    print(f"  Shape: {coherence.shape}")
    print(f"  Min: {np.nanmin(coherence):.3f}")
    print(f"  Max: {np.nanmax(coherence):.3f}")
    print(f"  Mean: {np.nanmean(coherence):.3f}")
    print(f"  Median: {np.nanmedian(coherence):.3f}")
    
    return coherence

# Example usage:
coherence = read_coherence(gunw_file, polarization='HH', posting='80m')

### 5.3 Wrapped Interferogram (20 m posting)

**Path:** `/science/LSAR/GUNW/grids/frequencyA/wrappedInterferogram/[HH/VV]/wrappedInterferogram`

- **Data Type:** Complex64 (float32 real + float32 imaginary)
- **Grid Spacing:** 20 m
- **Description:** Complex wrapped interferogram (multi-looked at 30 m in range-Doppler, geocoded at 20 m)
- **Phase Range:** -π to +π radians

In [ ]:
def read_wrapped_interferogram(filename, polarization='HH'):
    """
    Read and display wrapped interferogram
    """
    with h5py.File(filename, 'r') as f:
        base_path = f'/science/LSAR/GUNW/grids/frequencyA/wrappedInterferogram/{polarization}'
        
        # Read complex interferogram
        ifg_cpx = f[f'{base_path}/wrappedInterferogram'][:]
        
        # Convert to complex array if stored as compound type
        if ifg_cpx.dtype.names:
            ifg = ifg_cpx['r'] + 1j * ifg_cpx['i']
        else:
            ifg = ifg_cpx
        
        # Calculate phase and amplitude
        phase = np.angle(ifg)
        amplitude = np.abs(ifg)
        
        x_coords = f[f'{base_path}/xCoordinates'][:]
        y_coords = f[f'{base_path}/yCoordinates'][:]
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    # Phase
    im1 = axes[0].imshow(phase, cmap='hsv', vmin=-np.pi, vmax=np.pi, aspect='auto',
                         extent=[x_coords[0], x_coords[-1], y_coords[-1], y_coords[0]])
    axes[0].set_xlabel('Easting (m)')
    axes[0].set_ylabel('Northing (m)')
    axes[0].set_title(f'Wrapped Phase - {polarization}')
    cbar1 = plt.colorbar(im1, ax=axes[0], orientation='vertical', pad=0.02)
    cbar1.set_label('Phase (radians)', rotation=270, labelpad=20)
    
    # Amplitude
    im2 = axes[1].imshow(amplitude, cmap='gray', aspect='auto',
                         extent=[x_coords[0], x_coords[-1], y_coords[-1], y_coords[0]])
    axes[1].set_xlabel('Easting (m)')
    axes[1].set_ylabel('Northing (m)')
    axes[1].set_title(f'Amplitude - {polarization}')
    cbar2 = plt.colorbar(im2, ax=axes[1], orientation='vertical', pad=0.02)
    cbar2.set_label('Amplitude', rotation=270, labelpad=20)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nWrapped Interferogram Statistics ({polarization}):")
    print(f"  Shape: {ifg.shape}")
    print(f"  Mean amplitude: {np.nanmean(amplitude):.3f}")
    
    return ifg, phase, amplitude

# Example usage:
# ifg, phase, amplitude = read_wrapped_interferogram(gunw_file, polarization='HH')

### 5.4 Coherence Magnitude at 20 m posting

**Path:** `/science/LSAR/GUNW/grids/frequencyA/wrappedInterferogram/[HH/VV]/coherenceMagnitude`

- **Units:** Dimensionless (0-1)
- **Data Type:** float32
- **Grid Spacing:** 20 m
- **Description:** Higher resolution coherence magnitude corresponding to wrapped interferogram

## 6. Correction Layers

GUNW products include several atmospheric and geophysical correction layers. These corrections are **provided as separate layers but NOT applied to the data by default**. Users must apply them based on their analysis needs.

### 6.1 Ionospheric Phase Screen

**Path:** `/science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/[HH/VV]/ionospherePhaseScreen`

- **Units:** Radians
- **Data Type:** float32
- **Grid Spacing:** 80 m
- **Method:** 
  - Split-spectrum technique using frequencyA and frequencyB when available
  - Range split-spectrum on frequencyA alone during mode transitions
- **Uncertainty:** Provided in `ionospherePhaseScreenUncertainty`

In [ ]:
def read_ionosphere_correction(filename, polarization='HH'):
    """
    Read and display ionospheric phase screen
    """
    with h5py.File(filename, 'r') as f:
        base_path = f'/science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/{polarization}'
        
        iono = f[f'{base_path}/ionospherePhaseScreen'][:]
        iono_unc = f[f'{base_path}/ionospherePhaseScreenUncertainty'][:]
        
        x_coords = f[f'{base_path}/xCoordinates'][:]
        y_coords = f[f'{base_path}/yCoordinates'][:]
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    # Ionosphere
    im1 = axes[0].imshow(iono, cmap='RdBu_r', aspect='auto',
                         extent=[x_coords[0], x_coords[-1], y_coords[-1], y_coords[0]])
    axes[0].set_xlabel('Easting (m)')
    axes[0].set_ylabel('Northing (m)')
    axes[0].set_title('Ionospheric Phase Screen')
    cbar1 = plt.colorbar(im1, ax=axes[0], orientation='vertical', pad=0.02)
    cbar1.set_label('Phase (radians)', rotation=270, labelpad=20)
    
    # Uncertainty
    im2 = axes[1].imshow(iono_unc, cmap='hot', aspect='auto',
                         extent=[x_coords[0], x_coords[-1], y_coords[-1], y_coords[0]])
    axes[1].set_xlabel('Easting (m)')
    axes[1].set_ylabel('Northing (m)')
    axes[1].set_title('Ionospheric Phase Uncertainty')
    cbar2 = plt.colorbar(im2, ax=axes[1], orientation='vertical', pad=0.02)
    cbar2.set_label('Uncertainty (radians)', rotation=270, labelpad=20)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nIonospheric Correction Statistics:")
    print(f"  Phase Screen:")
    print(f"    Min: {np.nanmin(iono):.3f} rad")
    print(f"    Max: {np.nanmax(iono):.3f} rad")
    print(f"    Mean: {np.nanmean(iono):.3f} rad")
    print(f"  Uncertainty:")
    print(f"    Mean: {np.nanmean(iono_unc):.3f} rad")
    print(f"    Max: {np.nanmax(iono_unc):.3f} rad")
    
    return iono, iono_unc

# Example usage:
iono, iono_unc = read_ionosphere_correction(gunw_file, polarization='HH')

### 6.2 Tropospheric Corrections

Tropospheric phase delays are provided as separate lookup tables:

**Paths:**
- Dry (Hydrostatic): `/science/LSAR/GUNW/metadata/radarGrid/troposphereHydrostatic`
- Wet: `/science/LSAR/GUNW/metadata/radarGrid/troposphereWet`

- **Units:** Radians
- **Format:** 3D metadata cube (height-dependent)
- **Source:** From ECMWF High Res Forecast 
- **Components:**
  - **Hydrostatic delay**: Due to atmospheric pressure
  - **Wet delay**: Due to water vapor content

### 6.3 Solid Earth Tide Correction

**Path:** `/science/LSAR/GUNW/metadata/radarGrid/solidEarthTide`

- **Units:** Radians
- **Format:** 3D metadata cube
- **Description:** Phase contribution from solid Earth tides (gravitational deformation)
- **Model:** Based on astronomical ephemeris

In [ ]:
def apply_corrections(phase, ionosphere=None, troposphere_wet=None, 
                     troposphere_dry=None, solid_earth_tide=None):
    """
    Apply atmospheric and geophysical corrections to unwrapped phase
    
    Note: All corrections should be interpolated to the same grid as phase
    """
    corrected_phase = phase.copy()
    
    if ionosphere is not None:
        corrected_phase -= ionosphere
        print("Applied ionospheric correction")
    
    if troposphere_wet is not None:
        corrected_phase -= troposphere_wet
        print("Applied wet tropospheric correction")
    
    if troposphere_dry is not None:
        corrected_phase -= troposphere_dry
        print("Applied dry tropospheric correction")
    
    if solid_earth_tide is not None:
        corrected_phase -= solid_earth_tide
        print("Applied solid Earth tide correction")
    
    return corrected_phase

# Example usage:
# corrected_phase = apply_corrections(phase, ionosphere=iono)

## 7. Unwrapping Quality Metric: Connected Components

**Path:** `/science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/[HH/VV]/connectedComponents`

- **Data Type:** uint32
- **Grid Spacing:** 80 m
- **Description:** Label map identifying spatially connected regions in the unwrapped phase

### Interpretation:

- Each unique integer value represents a separate connected component
- Pixels with the **same label value** are spatially connected and can be compared directly
- Pixels with **different label values** may have unwrapping ambiguities (2π * N offset)
- **Larger components** generally indicate better unwrapping quality
- **Component value 0** typically indicates invalid or masked pixels

In [ ]:
def read_connected_components(filename, polarization='HH'):
    """
    Read and analyze connected components
    """
    with h5py.File(filename, 'r') as f:
        base_path = f'/science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/{polarization}'
        
        cc = f[f'{base_path}/connectedComponents'][:]
        x_coords = f[f'{base_path}/xCoordinates'][:]
        y_coords = f[f'{base_path}/yCoordinates'][:]
    
    # Analyze components
    unique_components = np.unique(cc[cc > 0])  # Exclude 0 (invalid)
    n_components = len(unique_components)
    
    # Calculate component sizes
    component_sizes = []
    for comp_id in unique_components:
        size = np.sum(cc == comp_id)
        component_sizes.append(size)
    
    component_sizes = np.array(component_sizes)
    largest_comp = unique_components[np.argmax(component_sizes)]
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    # Connected components map
    im1 = axes[0].imshow(cc, cmap='tab20', aspect='auto',
                         extent=[x_coords[0], x_coords[-1], y_coords[-1], y_coords[0]])
    axes[0].set_xlabel('Easting (m)')
    axes[0].set_ylabel('Northing (m)')
    axes[0].set_title(f'Connected Components ({n_components} components)')
    
    # Histogram of component sizes
    axes[1].hist(component_sizes, bins=50, edgecolor='black')
    axes[1].set_xlabel('Component Size (pixels)')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Distribution of Component Sizes')
    axes[1].set_yscale('log')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print(f"\nConnected Components Statistics:")
    print(f"  Total components: {n_components}")
    print(f"  Largest component ID: {largest_comp}")
    print(f"  Largest component size: {component_sizes.max()} pixels ({100*component_sizes.max()/cc.size:.1f}%)")
    print(f"  Mean component size: {component_sizes.mean():.1f} pixels")
    print(f"  Median component size: {np.median(component_sizes):.1f} pixels")
    print(f"  Components > 1000 pixels: {np.sum(component_sizes > 1000)}")
    
    return cc, component_sizes

# Example usage:
# cc, sizes = read_connected_components(gunw_file, polarization='HH')

## 8. Mask Layer

**Path:** `/science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/mask`

- **Data Type:** uint8
- **Grid Spacing:** 80 m
- **Description:** Pixel classification mask

### Mask Values:

The mask is a bit-encoded field with the following values:

- **Bit 0 (value 1)**: Layover
- **Bit 1 (value 2)**: Shadow
- **Bit 2 (value 4)**: Land
- **Bit 3 (value 8)**: Water

Values can be combined (e.g., 5 = layover + land)

In [ ]:
def read_mask(filename):
    """
    Read and decode mask layer
    """
    with h5py.File(filename, 'r') as f:
        mask_path = '/science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/mask'
        mask = f[mask_path][:]
        
        # Get polarization for coordinates (use first available)
        freq_group = f['/science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram']
        pols = [key for key in freq_group.keys() if key in ['HH', 'VV', 'HV', 'VH']]
        pol = pols[0] if pols else 'HH'
        
        x_coords = f[f'/science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/{pol}/xCoordinates'][:]
        y_coords = f[f'/science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/{pol}/yCoordinates'][:]
    
    # Decode mask
    layover = (mask & 1) > 0
    shadow = (mask & 2) > 0
    land = (mask & 4) > 0
    water = (mask & 8) > 0
    
    # Plot
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    extent = [x_coords[0], x_coords[-1], y_coords[-1], y_coords[0]]
    
    # Layover
    axes[0,0].imshow(layover, cmap='Reds', aspect='auto', extent=extent)
    axes[0,0].set_title(f'Layover ({100*layover.sum()/layover.size:.2f}%)')
    axes[0,0].set_xlabel('Easting (m)')
    axes[0,0].set_ylabel('Northing (m)')
    
    # Shadow
    axes[0,1].imshow(shadow, cmap='Blues', aspect='auto', extent=extent)
    axes[0,1].set_title(f'Shadow ({100*shadow.sum()/shadow.size:.2f}%)')
    axes[0,1].set_xlabel('Easting (m)')
    axes[0,1].set_ylabel('Northing (m)')
    
    # Land
    axes[1,0].imshow(land, cmap='Greens', aspect='auto', extent=extent)
    axes[1,0].set_title(f'Land ({100*land.sum()/land.size:.2f}%)')
    axes[1,0].set_xlabel('Easting (m)')
    axes[1,0].set_ylabel('Northing (m)')
    
    # Water
    axes[1,1].imshow(water, cmap='Blues', aspect='auto', extent=extent)
    axes[1,1].set_title(f'Water ({100*water.sum()/water.size:.2f}%)')
    axes[1,1].set_xlabel('Easting (m)')
    axes[1,1].set_ylabel('Northing (m)')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nMask Statistics:")
    print(f"  Layover pixels: {layover.sum()} ({100*layover.sum()/layover.size:.2f}%)")
    print(f"  Shadow pixels: {shadow.sum()} ({100*shadow.sum()/shadow.size:.2f}%)")
    print(f"  Land pixels: {land.sum()} ({100*land.sum()/land.size:.2f}%)")
    print(f"  Water pixels: {water.sum()} ({100*water.sum()/water.size:.2f}%)")
    
    return mask, layover, shadow, land, water

# Example usage:
# mask, layover, shadow, land, water = read_mask(gunw_file)

## 9. Pixel Offsets

Pixel offsets are measurements of ground displacement in two directions, derived from incoherent cross-correlation.

### 9.1 Along-Track Offset

**Path:** `/science/LSAR/GUNW/grids/frequencyA/pixelOffsets/[HH/VV]/alongTrackOffset`

- **Units:** Meters
- **Data Type:** float32
- **Grid Spacing:** 80 m
- **Description:** Displacement in satellite flight direction

### 9.2 Slant Range Offset

**Path:** `/science/LSAR/GUNW/grids/frequencyA/pixelOffsets/[HH/VV]/slantRangeOffset`

- **Units:** Meters
- **Data Type:** float32
- **Grid Spacing:** 80 m
- **Description:** Displacement in radar line-of-sight direction

### 9.3 Correlation Surface Peak

**Path:** `/science/LSAR/GUNW/grids/frequencyA/pixelOffsets/[HH/VV]/correlationSurfacePeak`

- **Units:** Dimensionless (0-1)
- **Description:** Quality metric for offset measurements

In [ ]:
def read_pixel_offsets(filename, polarization='HH'):
    """
    Read and display pixel offsets
    """
    with h5py.File(filename, 'r') as f:
        base_path = f'/science/LSAR/GUNW/grids/frequencyA/pixelOffsets/{polarization}'
        
        along_track = f[f'{base_path}/alongTrackOffset'][:]
        slant_range = f[f'{base_path}/slantRangeOffset'][:]
        corr_peak = f[f'{base_path}/correlationSurfacePeak'][:]
        
        x_coords = f[f'{base_path}/xCoordinates'][:]
        y_coords = f[f'{base_path}/yCoordinates'][:]
    
    # Plot
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    extent = [x_coords[0], x_coords[-1], y_coords[-1], y_coords[0]]
    
    # Along-track offset
    im1 = axes[0,0].imshow(along_track, cmap='RdBu_r', aspect='auto', extent=extent,
                          vmin=np.nanpercentile(along_track, 5),
                          vmax=np.nanpercentile(along_track, 95))
    axes[0,0].set_title('Along-Track Offset')
    axes[0,0].set_xlabel('Easting (m)')
    axes[0,0].set_ylabel('Northing (m)')
    cbar1 = plt.colorbar(im1, ax=axes[0,0], orientation='vertical', pad=0.02)
    cbar1.set_label('Offset (m)', rotation=270, labelpad=20)
    
    # Slant range offset
    im2 = axes[0,1].imshow(slant_range, cmap='RdBu_r', aspect='auto', extent=extent,
                          vmin=np.nanpercentile(slant_range, 5),
                          vmax=np.nanpercentile(slant_range, 95))
    axes[0,1].set_title('Slant Range Offset')
    axes[0,1].set_xlabel('Easting (m)')
    axes[0,1].set_ylabel('Northing (m)')
    cbar2 = plt.colorbar(im2, ax=axes[0,1], orientation='vertical', pad=0.02)
    cbar2.set_label('Offset (m)', rotation=270, labelpad=20)
    
    # Correlation peak
    im3 = axes[1,0].imshow(corr_peak, cmap='viridis', vmin=0, vmax=1, aspect='auto', extent=extent)
    axes[1,0].set_title('Correlation Surface Peak')
    axes[1,0].set_xlabel('Easting (m)')
    axes[1,0].set_ylabel('Northing (m)')
    cbar3 = plt.colorbar(im3, ax=axes[1,0], orientation='vertical', pad=0.02)
    cbar3.set_label('Correlation', rotation=270, labelpad=20)
    
    # Offset magnitude
    offset_mag = np.sqrt(along_track**2 + slant_range**2)
    im4 = axes[1,1].imshow(offset_mag, cmap='hot', aspect='auto', extent=extent,
                          vmin=0, vmax=np.nanpercentile(offset_mag, 95))
    axes[1,1].set_title('Offset Magnitude')
    axes[1,1].set_xlabel('Easting (m)')
    axes[1,1].set_ylabel('Northing (m)')
    cbar4 = plt.colorbar(im4, ax=axes[1,1], orientation='vertical', pad=0.02)
    cbar4.set_label('Magnitude (m)', rotation=270, labelpad=20)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nPixel Offset Statistics:")
    print(f"  Along-Track Offset:")
    print(f"    Mean: {np.nanmean(along_track):.3f} m")
    print(f"    Std: {np.nanstd(along_track):.3f} m")
    print(f"  Slant Range Offset:")
    print(f"    Mean: {np.nanmean(slant_range):.3f} m")
    print(f"    Std: {np.nanstd(slant_range):.3f} m")
    print(f"  Correlation Peak:")
    print(f"    Mean: {np.nanmean(corr_peak):.3f}")
    print(f"    High quality (>0.5): {100*np.sum(corr_peak>0.5)/corr_peak.size:.1f}%")
    
    return along_track, slant_range, corr_peak

# Example usage:
# az_off, rg_off, corr = read_pixel_offsets(gunw_file, polarization='HH')

## 10. Metadata Cube

**Path:** `/science/LSAR/GUNW/metadata/radarGrid/`

Metadata cubes are 3D arrays that provide radar geometry information organized over a geographic grid with height dependence.

### Key Metadata Cubes:

1. **incidenceAngle**: Local incidence angle (degrees)
2. **slantRange**: Slant range distance (m)
3. **zeroDopplerAzimuthTime**: Azimuth time (seconds)
4. **parallelBaseline**: Parallel baseline component (m)
5. **perpendicularBaseline**: Perpendicular baseline component (m)
6. **losUnitVectorX/Y**: Line-of-sight unit vector components
7. **alongTrackUnitVectorX/Y**: Along-track unit vector components
8. **elevationAngle**: Elevation angle (degrees)
9. **groundTrackVelocity**: Ground track velocity (m/s)

### Dimensions:
- **X**: Geographic easting
- **Y**: Geographic northing
- **Z**: Height above ellipsoid

### Usage:
Metadata cubes allow height-dependent interpolation of radar geometry parameters for any point in the scene.
A digital elevation model that covers the NISAR product is required to interpolate the metadata cube. We should make sure the provided DEM is exactly on the same grid (projection system) of the GUNW product.

Let's first extract the geolocation information of the GUNW product. We will next use these information to reproject the DEM to the grid of the GUNW product.

In [ ]:
def read_gunw_grid_info(gunw_file, polarization='HH'):
    """
    Read grid information from GUNW product
    
    Parameters:
    -----------
    gunw_file : str
        Path to GUNW HDF5 file
    polarization : str
        Polarization (HH or VV)
    
    Returns:
    --------
    grid_info : dict
        Dictionary containing:
        - x_coords: X coordinates (easting)
        - y_coords: Y coordinates (northing)
        - epsg: EPSG code
        - projection_wkt: Projection in WKT format
        - bbox: Bounding box (xmin, ymin, xmax, ymax)
        - shape: Grid shape (ny, nx)
    """
    with h5py.File(gunw_file, 'r') as f:
        base_path = f'/science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/{polarization}'
        
        # Read coordinates
        x_coords = f[f'{base_path}/xCoordinates'][:]
        y_coords = f[f'{base_path}/yCoordinates'][:]
        
        # Read projection
        projection = f[f'{base_path}/projection'][()]
        #if isinstance(projection, bytes):
        #    projection = projection.decode('utf-8')
        
        # Read sample data to get shape
        sample_data = f[f'{base_path}/unwrappedPhase']
        shape = sample_data.shape
    
    # Extract EPSG code from projection
    #srs = osr.SpatialReference()
    #srs.ImportFromWkt(projection)
    #epsg = None
    #if srs.GetAuthorityName(None) == 'EPSG':
    #    epsg = int(srs.GetAuthorityCode(None))
    epsg = int(projection)
    # Calculate bounding box
    bbox = {
        'xmin': x_coords.min(),
        'xmax': x_coords.max(),
        'ymin': y_coords.min(),
        'ymax': y_coords.max()
    }
    
    grid_info = {
        'x_coords': x_coords,
        'y_coords': y_coords,
        'epsg': epsg,
        'projection_wkt': projection,
        'bbox': bbox,
        'shape': shape,
        'x_spacing': np.median(np.diff(x_coords)),
        'y_spacing': np.median(np.diff(y_coords))
    }
    
    print("GUNW Grid Information:")
    print(f"  EPSG Code: {epsg}")
    print(f"  Grid Shape: {shape}")
    print(f"  X Range: {bbox['xmin']:.2f} to {bbox['xmax']:.2f}")
    print(f"  Y Range: {bbox['ymin']:.2f} to {bbox['ymax']:.2f}")
    print(f"  X Spacing: {grid_info['x_spacing']:.2f} m")
    print(f"  Y Spacing: {grid_info['y_spacing']:.2f} m")
    
    return grid_info

# Example usage:
gunw_file = 'NISAR_L2_PR_GUNW_024_013_D_071_025_4000_SH_20260627T024036_20260627T024114_20260709T024036_20260709T024113_P05023_N_F_J_001.h5'
grid_info = read_gunw_grid_info(gunw_file, polarization='HH')
grid_info

The following function ensures that the provided DEM is on the exact grid of the GUNW product. 

In [ ]:
from osgeo import gdal

def reproject_dem_to_bbox(input_dem_path, output_dem_path, target_epsg, bbox, target_res):
    """
    Reprojects a DEM to a specific EPSG code, bounding box grid, and resolution.
    
    Parameters:
    - input_dem_path (str): Path to the original DEM file.
    - output_dem_path (str): Path where the reprojected DEM will be saved.
    - target_epsg (int): The EPSG code of the target coordinate system (e.g., 4326).
    - bbox (list/tuple): Target bounding box in the target EPSG format: [minx, miny, maxx, maxy].
    - target_res (float/tuple): Target pixel size. Use a single float for square pixels, 
                                 or a tuple (x_res, y_res) for different dimensions.
    """
    # Open the source dataset to read metadata
    src_ds = gdal.Open(input_dem_path)
    if src_ds is None:
        raise FileNotFoundError(f"Could not open input DEM at {input_dem_path}")
        
    # Unpack resolution if it is a tuple
    if isinstance(target_res, (list, tuple)):
        x_res, y_res = target_res
    else:
        x_res, y_res = target_res, target_res

    # Define warp options
    warp_options = gdal.WarpOptions(
        dstSRS=f'EPSG:{target_epsg}',
        outputBounds=bbox,              # [minx, miny, maxx, maxy]
        xRes=x_res,
        yRes=y_res,
        resampleAlg=gdal.GRIORA_Bilinear, # Best resampling algorithm for continuous DEM data
        dstNodata=-9999                 # Sets a standard NoData value for the output grid
    )

    # Execute the warping process
    gdal.Warp(output_dem_path, src_ds, options=warp_options)
    
    # Close the dataset safely
    src_ds = None
    print(f"Successfully reprojected DEM saved to: {output_dem_path}")

# --- Example Usage ---
# my_bbox = [xmin, ymin, xmax, ymax] in EPSG:32632

input_dem = "NISAR_dem_0.tiff"
gunw_dem = "NISAR_GUNW_DEM.tif"
output_bbox = [grid_info['bbox']['xmin'], grid_info['bbox']['ymin'], grid_info['bbox']['xmax'], grid_info['bbox']['ymax']]
target_res = grid_info['x_spacing']
reproject_dem_to_bbox(input_dem, gunw_dem, grid_info['epsg'], output_bbox, target_res)


Now the following function interpolates any GUNW three dimensional metadata cube to a two dimensional dataset.

In [ ]:
import numpy as np
import h5py
from osgeo import gdal
from scipy.interpolate import RegularGridInterpolator

def interpolate_nisar_cube_to_dem(gunw_path, cube_dataset, dem_path, output_path):
    """
    Interpolates a 3D NISAR L2 metadata cube layer to a 2D grid matching a DEM's
    exact coordinates and local topography heights.
    
    Parameters:
    - gunw_path (str): Path to the NISAR L2 GUNW HDF5 file.
    - cube_dataset (str): Name of the 3D dataset to be evaluated on the DEM grid
    - dem_path (str): Path to the target reprojected DEM (GeoTIFF format).
    - output_path (str): Path to save the interpolated 2D output layer.
    """
    radar_cube_path = "/science/LSAR/GUNW/metadata/radarGrid" 
    # 1. Read the DEM data and its geographic grid
    dem_ds = gdal.Open(dem_path)
    if dem_ds is None:
        raise FileNotFoundError(f"Could not open DEM at {dem_path}")
        
    geotransform = dem_ds.GetGeoTransform()
    dem_width = dem_ds.RasterXSize
    dem_height = dem_ds.RasterYSize
    dem_data = dem_ds.ReadAsArray() # Matrix of heights (Z)
    
    # Generate 1D coordinate vectors matching the pixel centers of the DEM
    dem_x = geotransform[0] + (np.arange(dem_width) + 0.5) * geotransform[1]
    dem_y = geotransform[3] + (np.arange(dem_height) + 0.5) * geotransform[5]

    cube_group_path = f"{radar_cube_path}/{cube_dataset}"
    cube_x_path = f"{radar_cube_path}/xCoordinates"
    cube_y_path = f"{radar_cube_path}/yCoordinates"
    cube_z_path = f"{radar_cube_path}/heightAboveEllipsoid"
    # 2. Extract the NISAR 3D metadata cube and its axis coordinate definitions
    with h5py.File(gunw_path, 'r') as h5:
        cube_ds = h5[cube_group_path]
        cube_data = cube_ds[:]  # Shape typically expected: (len(cube_z), len(cube_y), len(cube_x))
        
        # Read the coordinate axes vectors from HDF5 attributes or parallel coordinate datasets
        cube_x = h5[cube_x_path][:]
        cube_y = h5[cube_y_path][:]
        cube_z = h5[cube_z_path][:]
        
    # 3. Create the 3D Regular Grid Interpolator
    # Ensure dimensions match your cube structure. If data is (Z, Y, X), order the axes accordingly
    interpolator = RegularGridInterpolator(
        (cube_z, cube_y, cube_x), 
        cube_data, 
        method='linear', 
        bounds_error=False, 
        fill_value=np.nan
    )
    
    # 4. Construct the high-resolution interpolation grid query
    # Meshgrid creates 2D arrays matching the DEM footprint
    dem_X, dem_Y = np.meshgrid(dem_x, dem_y)
    
    # Flatten everything to pass as a N x 3 column matrix: [Z_dem, Y_dem, X_dem]
    query_points = np.column_stack((
        dem_data.ravel(),  # The local height profile dictates the Z slice queried
        dem_Y.ravel(), 
        dem_X.ravel()
    ))
    
    # 5. Execute 3D Interpolation
    print("Interpolating 3D cube onto DEM target grid...")
    interpolated_flat = interpolator(query_points)
    
    # Reshape back to the original 2D DEM shape
    interpolated_2d = interpolated_flat.reshape((dem_height, dem_width))
    
    # 6. Save the output to a GeoTIFF maintaining spatial metadata
    driver = gdal.GetDriverByName("GTiff")
    out_ds = driver.Create(output_path, dem_width, dem_height, 1, gdal.GDT_Float32)
    out_ds.SetGeoTransform(geotransform)
    out_ds.SetProjection(dem_ds.GetProjection())
    
    out_band = out_ds.GetRasterBand(1)
    out_band.WriteArray(interpolated_2d)
    out_band.SetNoDataValue(np.nan)
    
    # Flush buffers
    out_band.FlushCache()
    dem_ds = None
    out_ds = None
    print(f"Interpolated layer successfully saved to {output_path}")

    return interpolated_2d

## Tropospheric phase delay

In [ ]:
tropo_phase_wet_file = "tropo_phase_wet.tif"
tropo_phase_hydro_file = "tropo_phase_hydro.tif"
tropo_phase_wet = interpolate_nisar_cube_to_dem(gunw_file, "wetTroposphericPhaseScreen", gunw_dem, tropo_phase_wet_file)
tropo_phase_hydro = interpolate_nisar_cube_to_dem(gunw_file, "hydrostaticTroposphericPhaseScreen", gunw_dem, tropo_phase_hydro_file)
tropo_phae = tropo_phase_wet + tropo_phase_hydro

In [ ]:
plt.imshow(tropo_phae)
plt.colorbar()

## Solid earth tide

In [ ]:
solid_earth_tide_file = "solid_earth_tide.tif"
solid_earth_tide = interpolate_nisar_cube_to_dem(gunw_file, "slantRangeSolidEarthTidesPhase", gunw_dem, solid_earth_tide_file)

## 11. Phase Sign Convention

### IMPORTANT: Understanding Phase Signs

NISAR GUNW products follow this **phase sign convention**:

- **POSITIVE PHASE (+)**: Range increase → Moving AWAY from sensor (subsidence, extension)
- **NEGATIVE PHASE (-)**: Range decrease → Moving TOWARD sensor (uplift, compression)

### Mathematical Definition:

$$\phi = \phi_{\text{secondary}} - \phi_{\text{reference}}$$

Where:
- $\phi$: Interferometric phase
- $\phi_{\text{reference}}$: Phase from earlier acquisition
- $\phi_{\text{secondary}}$: Phase from later acquisition

### Relationship to Displacement:

$$\Delta R = -\frac{\lambda}{4\pi} \phi$$

Where:
- $\Delta R$: Line-of-sight displacement (positive = toward sensor)
- $\lambda$: Radar wavelength (L-band: ~24 cm)
- $\phi$: Interferometric phase

### Examples:

1. **Subsidence (sinking)**:
   - Ground moves DOWN
   - Range to sensor INCREASES
   - Phase is POSITIVE

2. **Uplift (rising)**:
   - Ground moves UP
   - Range to sensor DECREASES
   - Phase is NEGATIVE



In [ ]:
def phase_to_displacement(phase, wavelength, look_angle=None):
    """
    Convert interferometric phase to line-of-sight displacement
    
    Parameters:
    -----------
    phase : ndarray
        Interferometric phase in radians
    wavelength : float
        Radar wavelength in meters (L-band default: 0.24 m)
    look_angle : ndarray, optional
        Look angle in degrees for vertical displacement conversion
    
    Returns:
    --------
    los_displacement : ndarray
        Line-of-sight displacement in meters (positive = toward sensor)
    """
    # LOS displacement
    los_displacement = -(wavelength / (4 * np.pi)) * phase

    return los_displacement

# Example usage:
# los_disp = phase_to_displacement(phase, wavelength=0.24)

# With vertical component:
# los_disp, vert_disp = phase_to_displacement(phase, wavelength=0.24, look_angle=35)

## 12. Processing Information

**Path:** `/science/LSAR/GUNW/metadata/processingInformation/`

This group contains comprehensive information about how the GUNW product was generated.

### 12.1 Input Files

**Path:** `/science/LSAR/GUNW/metadata/processingInformation/inputs/`

Contains:
- Reference RSLC filename
- Secondary RSLC filename
- DEM source and description
- Orbit files
- Configuration files

In [ ]:
def print_processing_inputs(filename):
    """
    Print input files used for processing
    """
    with h5py.File(filename, 'r') as f:
        inputs = f['/science/LSAR/GUNW/metadata/processingInformation/inputs']
        
        print("\nProcessing Inputs:")
        print("=" * 80)
        
        # List all datasets in inputs group
        for key in inputs.keys():
            value = inputs[key][()]
            if isinstance(value, bytes):
                value = value.decode('utf-8')
            print(f"  {key}:")
            print(f"    {value}")

# Example usage:
print_processing_inputs(gunw_file)

### 12.2 Processing Parameters

**Path:** `/science/LSAR/GUNW/metadata/processingInformation/parameters/`

Contains parameters for:
- **common**: Combined reference and secondary parameters
- **reference**: Reference RSLC parameters
- **secondary**: Secondary RSLC parameters
- **interferogram**: Interferogram formation parameters (looks, bandwidths)
- **ionosphere**: Ionosphere estimation parameters
- **pixelOffsets**: Offset tracking parameters (window sizes, spacing)
- **geocoding**: Geocoding parameters and applied corrections

In [ ]:
def print_processing_parameters(filename):
    """
    Print key processing parameters
    """
    with h5py.File(filename, 'r') as f:
        params = f['/science/LSAR/GUNW/metadata/processingInformation/parameters']
        
        print("\nProcessing Parameters:")
        print("=" * 80)
        
        # Interferogram parameters
        if 'interferogram' in params:
            ifg_params = params['interferogram/frequencyA']
            print("\nInterferogram Formation:")
            for key in ifg_params.keys():
                value = ifg_params[key][()]
                print(f"  {key}: {value}")
        
        # Geocoding parameters
        if 'geocoding' in params:
            geo_params = params['geocoding']
            print("\nGeocoding:")
            for key in geo_params.keys():
                value = geo_params[key][()]
                if isinstance(value, bytes):
                    value = value.decode('utf-8')
                print(f"  {key}: {value}")

# Example usage:
# print_processing_parameters(gunw_file)

### 12.3 Algorithm Information

**Path:** `/science/LSAR/GUNW/metadata/processingInformation/algorithms/`

Documents algorithms used for:
- **coregistration**: Image alignment methods
- **interferogramFormation**: Phase and coherence computation
- **unwrapping**: Phase unwrapping algorithm
- **ionosphereEstimation**: Ionosphere correction methods
- **geocoding**: Projection and interpolation methods

In [ ]:
def print_algorithms(filename):
    """
    Print algorithms used in processing
    """
    with h5py.File(filename, 'r') as f:
        algorithms = f['/science/LSAR/GUNW/metadata/processingInformation/algorithms']
        
        print("\nProcessing Algorithms:")
        print("=" * 80)
        
        def print_group(group, indent=0):
            for key in group.keys():
                item = group[key]
                if isinstance(item, h5py.Group):
                    print("  " * indent + f"{key}:")
                    print_group(item, indent + 1)
                else:
                    value = item[()]
                    if isinstance(value, bytes):
                        value = value.decode('utf-8')
                    print("  " * indent + f"{key}: {value}")
        
        print_group(algorithms, indent=1)

# Example usage:
# print_algorithms(gunw_file)

### 12.4 Run Configuration

**Path:** `/science/LSAR/GUNW/metadata/processingInformation/parameters/runConfigurationContents`

Complete run configuration file (YAML format) used for processing, including all parameters and input files.

In [ ]:
def extract_run_config(filename, output_file='runconfig.yaml'):
    """
    Extract and save run configuration
    """
    with h5py.File(filename, 'r') as f:
        config_path = '/science/LSAR/GUNW/metadata/processingInformation/parameters/runConfigurationContents'
        
        if config_path in f:
            config = f[config_path][()]
            if isinstance(config, bytes):
                config = config.decode('utf-8')
            
            # Save to file
            with open(output_file, 'w') as out:
                out.write(config)
            
            print(f"Run configuration saved to: {output_file}")
            print(f"\nFirst 50 lines:\n")
            print('\n'.join(config.split('\n')[:50]))
        else:
            print("Run configuration not found in product")

# Example usage:
# extract_run_config(gunw_file)

## 13. QA Report and Browse

Quality assurance products are typically distributed alongside GUNW products:

### 13.1 Browse Images

- **Format**: PNG or GeoTIFF
- **Content**: Quick-look images of key datasets
  - Unwrapped phase
  - Coherence
  - Connected components
- **Purpose**: Rapid visual quality assessment

### 13.2 QA Report

- **Format**: JSON or XML
- **Content**: Statistical summaries and quality metrics
  - Mean coherence
  - Percentage of valid pixels
  - Layover/shadow statistics
  - Unwrapping quality metrics
  - Processing warnings/errors

In [ ]:
def create_browse_images(filename, output_dir='.', polarization='HH'):
    """
    Create browse images from GUNW product
    """
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    with h5py.File(filename, 'r') as f:
        # Unwrapped phase
        phase_path = f'/science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/{polarization}/unwrappedPhase'
        phase = f[phase_path][:]
        
        # Coherence
        coh_path = f'/science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/{polarization}/coherenceMagnitude'
        coherence = f[coh_path][:]
        
        # Connected components
        cc_path = f'/science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/{polarization}/connectedComponents'
        cc = f[cc_path][:]
    
    # Create browse images
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Phase
    im1 = axes[0].imshow(phase, cmap='jet', aspect='auto')
    axes[0].set_title('Unwrapped Phase')
    axes[0].axis('off')
    plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)
    
    # Coherence
    im2 = axes[1].imshow(coherence, cmap='gray', vmin=0, vmax=1, aspect='auto')
    axes[1].set_title('Coherence')
    axes[1].axis('off')
    plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)
    
    # Connected components
    im3 = axes[2].imshow(cc, cmap='tab20', aspect='auto')
    axes[2].set_title('Connected Components')
    axes[2].axis('off')
    plt.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04)
    
    plt.tight_layout()
    
    # Save
    output_file = os.path.join(output_dir, f'GUNW_browse_{polarization}.png')
    plt.savefig(output_file, dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"Browse image saved to: {output_file}")

# Example usage:
# create_browse_images(gunw_file, output_dir='./browse', polarization='HH')

## Summary and Best Practices

### Key Takeaways:

1. **GUNW products are geocoded L2 products** containing unwrapped interferometric phase and associated data

2. **Two resolution grids**:
   - 80 m: Unwrapped phase, coherence, corrections, offsets
   - 20 m: Wrapped interferogram and coherence

3. **Corrections are provided but NOT applied** - users must apply based on their needs

4. **Phase sign convention**:
   - Positive = range increase = moving away
   - Negative = range decrease = moving toward

5. **Quality indicators**:
   - Coherence magnitude (0-1)
   - Connected components (unwrapping reliability)
   - Correlation surface peak (offset quality)

### Recommended Workflow:

1. **Data Access**: Download from ASF DAAC
2. **Exploration**: Examine HDF5 structure and metadata
3. **Quality Check**: Review coherence and connected components
4. **Masking**: Apply layover/shadow/water masks as needed
5. **Corrections**: Apply atmospheric and geometric corrections
6. **Interpretation**: Convert phase to displacement using sign convention
7. **Analysis**: Perform time series, velocity estimation, or deformation modeling

### Additional Resources:

- **Product Specification**: JPL D-102272
- **Algorithm Theoretical Basis**: JPL D-95677
- **ASF DAAC**: https://search.asf.alaska.edu/
- **NISAR Mission**: https://nisar.jpl.nasa.gov/
- **HDF5 Documentation**: https://portal.hdfgroup.org/display/HDF5/HDF5

## Appendix: Complete Example Workflow

In [ ]:
def complete_gunw_analysis(gunw_file, polarization='HH'):
    """
    Complete workflow for GUNW product analysis
    """
    print("=" * 80)
    print("NISAR GUNW Product Analysis")
    print("=" * 80)
    
    # 1. Product identification
    print("\n1. Product Identification")
    print_global_attributes(gunw_file)
    print_identification_info(gunw_file)
    
    # 2. Read main datasets
    print("\n2. Reading Main Datasets")
    phase, x, y = read_unwrapped_phase(gunw_file, polarization)
    coherence = read_coherence(gunw_file, polarization, '80m')
    
    # 3. Quality assessment
    print("\n3. Quality Assessment")
    cc, sizes = read_connected_components(gunw_file, polarization)
    mask, layover, shadow, land, water = read_mask(gunw_file)
    
    # 4. Corrections
    print("\n4. Atmospheric Corrections")
    iono, iono_unc = read_ionosphere_correction(gunw_file, polarization)
    
    # 5. Apply corrections
    print("\n5. Applying Corrections")
    corrected_phase = apply_corrections(phase, ionosphere=iono)
    
    # 6. Convert to displacement
    print("\n6. Converting to Displacement")
    displacement = phase_to_displacement(corrected_phase, wavelength=0.24)
    
    print("\nDisplacement Statistics (LOS):")
    print(f"  Min: {np.nanmin(displacement):.3f} m")
    print(f"  Max: {np.nanmax(displacement):.3f} m")
    print(f"  Mean: {np.nanmean(displacement):.3f} m")
    print(f"  Std: {np.nanstd(displacement):.3f} m")
    
    # 7. Processing information
    print("\n7. Processing Information")
    print_processing_inputs(gunw_file)
    print_processing_parameters(gunw_file)
    
    # 8. Create browse images
    print("\n8. Creating Browse Images")
    create_browse_images(gunw_file, output_dir='./browse', polarization=polarization)
    
    print("\n" + "=" * 80)
    print("Analysis Complete!")
    print("=" * 80)
    
    return {
        'phase': phase,
        'coherence': coherence,
        'corrected_phase': corrected_phase,
        'displacement': displacement,
        'connected_components': cc,
        'mask': mask
    }

# Example usage:
# results = complete_gunw_analysis(gunw_file, polarization='HH')